In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np

import ast

In [3]:
credits = pd.read_csv('/content/drive/MyDrive/ML_project/project1_MovieRecommendationSystem/tmdb_5000_credits.csv')
movies = pd.read_csv('/content/drive/MyDrive/ML_project/project1_MovieRecommendationSystem/tmdb_5000_movies.csv')

In [4]:
# we are merge movie and credits dataset in to one
movies = movies.merge(credits,on='title')

In [5]:
# we choose featurse or colume only required or useful
movies = movies[['movie_id','title','overview','genres','keywords','cast','crew']]

In [6]:
def convert(text):
    L = []
    for i in ast.literal_eval(text):
        L.append(i['name'])
    return L

In [7]:
#this delete null value rows
movies.dropna(inplace=True)

In [8]:
# genres have string datatype we converted it into list

movies['genres'] = movies['genres'].apply(convert)
movies.head(1)

,movie_id,title,overview,genres,keywords,cast,crew
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...","[Action, Adventure, Fantasy, Science Fiction]","[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...","[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."


In [9]:
# Keyword have string datatype we converted it into list
movies['keywords'] = movies['keywords'].apply(convert)
movies.head(1)

,movie_id,title,overview,genres,keywords,cast,crew
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...","[Action, Adventure, Fantasy, Science Fiction]","[culture clash, future, space war, space colon...","[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."


In [10]:

import ast
# literal_eval convert String in to list
ast.literal_eval('[{"id": 28, "name": "Action"}, {"id": 12, "name": "Adventure"}, {"id": 14, "name": "Fantasy"}, {"id": 878, "name": "Science Fiction"}]')

[{'id': 28, 'name': 'Action'},
 {'id': 12, 'name': 'Adventure'},
 {'id': 14, 'name': 'Fantasy'},
 {'id': 878, 'name': 'Science Fiction'}]

In [11]:
# this convert string in to list of first 3 object and appen its name in to L list
def convert3(text):
    L = []
    counter = 0
    for i in ast.literal_eval(text):
        if counter < 3:
            L.append(i['name'])
        counter+=1
    return L

In [12]:
# By using convert3 function we retrive top 3 actor name from cast colume

movies['cast'] = movies['cast'].apply(convert)
movies.head(1)

,movie_id,title,overview,genres,keywords,cast,crew
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...","[Action, Adventure, Fantasy, Science Fiction]","[culture clash, future, space war, space colon...","[Sam Worthington, Zoe Saldana, Sigourney Weave...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."


In [13]:
#
movies['cast'] = movies['cast'].apply(lambda x:x[0:3])

In [14]:
# fencth director from crew colunm
def fetch_director(text):
    L = []
    for i in ast.literal_eval(text):
        if i['job'] == 'Director':
            L.append(i['name'])
    return L

In [15]:

movies['crew'] = movies['crew'].apply(fetch_director)

In [16]:
# overview colunm convert it in to list
movies['overview'] = movies['overview'].apply(lambda x:x.split())
movies.sample(5)

,movie_id,title,overview,genres,keywords,cast,crew
2610,7874,Black Snake Moan,"[A, God-fearing, bluesman, takes, to, a, wild,...",[Drama],"[southern usa, blues, military service, indepe...","[Samuel L. Jackson, Christina Ricci, Justin Ti...",[Craig Brewer]
871,8046,Gigli,"[Gigli, is, ordered, to, kidnap, the, psycholo...",[Drama],"[new york, mentally disabled, kidnapping, blac...","[Ben Affleck, Jennifer Lopez, Justin Bartha]",[Martin Brest]
256,262504,Allegiant,"[Beatrice, Prior, and, Tobias, Eaton, venture,...","[Adventure, Science Fiction]","[based on novel, revolution, dystopia, sequel,...","[Shailene Woodley, Theo James, Zoë Kravitz]",[Robert Schwentke]
3902,20337,Martin Lawrence Live: Runteldat,"[The, controversial, bad-boy, of, comedy, deli...",[Comedy],"[daily life, scandal, growing up, divorce]",[Martin Lawrence],"[Martin Lawrence, David Raynr]"
4456,367551,American Hero,"[Melvin,, a, reluctant, hero, who, is, far, fr...","[Action, Comedy, Science Fiction]",[],"[Stephen Dorff, Eddie Griffin, Bill Billions]",[Nick Love]


In [17]:
# remove spaces in names
def collapse(L):
    L1 = []
    for i in L:
        L1.append(i.replace(" ",""))
    return L1

In [18]:
movies['cast'] = movies['cast'].apply(collapse)
movies['crew'] = movies['crew'].apply(collapse)
movies['genres'] = movies['genres'].apply(collapse)
movies['keywords'] = movies['keywords'].apply(collapse)

In [19]:

movies.head(1)

,movie_id,title,overview,genres,keywords,cast,crew
0,19995,Avatar,"[In, the, 22nd, century,, a, paraplegic, Marin...","[Action, Adventure, Fantasy, ScienceFiction]","[cultureclash, future, spacewar, spacecolony, ...","[SamWorthington, ZoeSaldana, SigourneyWeaver]",[JamesCameron]


In [20]:
# all colums join in to one tag colunm
movies['tags'] = movies['overview'] + movies['genres'] + movies['keywords'] + movies['cast'] + movies['crew']

In [21]:
# now create new dataset with name new and id, title, tags colume only other remove
new = movies.drop(columns=['overview','genres','keywords','cast','crew'])
new.head()

,movie_id,title,tags
0,19995,Avatar,"[In, the, 22nd, century,, a, paraplegic, Marin..."
1,285,Pirates of the Caribbean: At World's End,"[Captain, Barbossa,, long, believed, to, be, d..."
2,206647,Spectre,"[A, cryptic, message, from, Bond’s, past, send..."
3,49026,The Dark Knight Rises,"[Following, the, death, of, District, Attorney..."
4,49529,John Carter,"[John, Carter, is, a, war-weary,, former, mili..."


In [22]:
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer(max_features=5000,stop_words='english')

In [23]:
new['tags'] = new['tags'].apply(lambda x: " ".join(x))
vector = cv.fit_transform(new['tags']).toarray()

In [24]:
vector.shape

(4806, 5000)

In [25]:
from sklearn.metrics.pairwise import cosine_similarity

In [26]:

similarity = cosine_similarity(vector)

In [27]:

similarity

array([[1.        , 0.08858079, 0.05905386, ..., 0.02478408, 0.02599376,
        0.        ],
       [0.08858079, 1.        , 0.06451613, ..., 0.02707652, 0.        ,
        0.        ],
       [0.05905386, 0.06451613, 1.        , ..., 0.02707652, 0.        ,
        0.        ],
       ...,
       [0.02478408, 0.02707652, 0.02707652, ..., 1.        , 0.07150969,
        0.0489116 ],
       [0.02599376, 0.        , 0.        , ..., 0.07150969, 1.        ,
        0.05129892],
       [0.        , 0.        , 0.        , ..., 0.0489116 , 0.05129892,
        1.        ]])

In [28]:
new[new['title'] == 'The Lego Movie'].index[0]

np.int64(744)

In [29]:

def recommend(movie):
    index = new[new['title'] == movie].index[0]
    distances = sorted(list(enumerate(similarity[index])),reverse=True,key = lambda x: x[1])
    for i in distances[1:6]:
        print(new.iloc[i[0]].title)

In [30]:

recommend('Gandhi')

Gandhi, My Father
The Wind That Shakes the Barley
A Passage to India
Guiana 1838
Ramanujan


In [32]:
import pickle

In [36]:
pickle.dump(new,open('/content/drive/MyDrive/ML_project/project1_MovieRecommendationSystem/movie_list.pkl','wb'))

pickle.dump(similarity,open('/content/drive/MyDrive/ML_project/project1_MovieRecommendationSystem/similarity.pkl','wb'))

In [34]:
import os
os.getcwd()

'/content'